# Surrogate Modeling Workshop - One-Click Environment Check

Choose **Runtime > Run all** in Google Colab or **Run > Run All Cells** in JupyterLab.

This notebook does **not** create or train a machine-learning model. It checks the Python runtime, required package imports, temporary CSV file I/O, Google Drive access in Colab, and access to key workshop websites.

In [ ]:
import os
import sys
import platform
import tempfile
import importlib
import urllib.request
from pathlib import Path
from datetime import datetime, timezone

RESULTS = []

def record(name, status, detail):
    status = status.upper()
    RESULTS.append({"check": name, "status": status, "detail": str(detail)})
    icon = {"PASS": "✅", "WARN": "⚠️", "FAIL": "❌"}.get(status, "•")
    print(f"{icon} {name}: {status} - {detail}")

IN_COLAB = "google.colab" in sys.modules
print("Environment check started:", datetime.now(timezone.utc).isoformat())
print("Execution environment:", "Google Colab" if IN_COLAB else "Local Jupyter/Python")

In [ ]:
print("Python executable:", sys.executable)
print("Python version:", sys.version.replace("\n", " "))
print("Operating system:", platform.platform())
print("Machine:", platform.machine())

if IN_COLAB:
    record("Python runtime", "PASS", f"Colab-managed Python {platform.python_version()}")
elif sys.version_info[:2] == (3, 11):
    record("Python runtime", "PASS", f"Local Python {platform.python_version()}")
else:
    record("Python runtime", "FAIL", f"Local Python 3.11 is required; found {platform.python_version()}")

In [ ]:
# Import checks only: no model is created or trained.
# Colab runtimes are temporary, so missing dependencies are installed there
# automatically. Local environments are never modified by this notebook.
PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scikit-learn": "sklearn",
    "TensorFlow": "tensorflow",
    "ONNX": "onnx",
    "skl2onnx": "skl2onnx",
}
PIP_SPECS = {
    "numpy": "numpy>=1.26,<2",
    "pandas": "pandas>=2.1,<3",
    "matplotlib": "matplotlib>=3.8,<4",
    "sklearn": "scikit-learn>=1.4,<2",
    "tensorflow": "tensorflow>=2.16,<2.19",
    "onnx": "onnx>=1.16,<2",
    "skl2onnx": "skl2onnx>=1.16,<2",
}

if IN_COLAB:
    import importlib.util
    import subprocess
    missing_specs = [spec for module, spec in PIP_SPECS.items() if importlib.util.find_spec(module) is None]
    if missing_specs:
        print("Installing missing packages into this temporary Colab runtime:", ", ".join(missing_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing_specs])
            importlib.invalidate_caches()
            record("Colab dependency installation", "PASS", "Installed missing packages")
        except Exception as exc:
            record("Colab dependency installation", "FAIL", f"{type(exc).__name__}: {exc}")
    else:
        record("Colab dependency installation", "PASS", "All packages were already available")

loaded = {}
for display_name, import_name in PACKAGES.items():
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "version not reported")
        loaded[import_name] = module
        record(f"Import {display_name}", "PASS", version)
    except Exception as exc:
        record(f"Import {display_name}", "FAIL", f"{type(exc).__name__}: {exc}")

In [ ]:
# Test the same basic CSV workflow used by the workshop notebooks.
if "pandas" in loaded and "numpy" in loaded:
    try:
        pd = loaded["pandas"]
        np = loaded["numpy"]
        test_frame = pd.DataFrame({
            "X1": np.array([1.0, 2.0, 3.0]),
            "X2": np.array([10.0, 20.0, 30.0]),
            "Y1": np.array([0.5, 1.0, 1.5]),
        })
        with tempfile.TemporaryDirectory() as temp_dir:
            csv_path = Path(temp_dir) / "workshop_environment_test.csv"
            test_frame.to_csv(csv_path, index=False)
            loaded_frame = pd.read_csv(csv_path)
            assert loaded_frame.shape == (3, 3)
            assert list(loaded_frame.columns) == ["X1", "X2", "Y1"]
        record("Temporary CSV read/write", "PASS", "Created, read, validated, and removed a test CSV")
    except Exception as exc:
        record("Temporary CSV read/write", "FAIL", f"{type(exc).__name__}: {exc}")
else:
    record("Temporary CSV read/write", "FAIL", "Requires working NumPy and pandas imports")

In [ ]:
# In Colab this prompts once for Drive authorization, then tests a workshop folder.
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        drive_root = Path("/content/drive/MyDrive")
        workshop_dir = drive_root / "Surrogate_Model_Workshop" / "environment_check"
        workshop_dir.mkdir(parents=True, exist_ok=True)
        marker = workshop_dir / "environment_check.txt"
        marker.write_text(
            "Google Drive write test completed at " + datetime.now(timezone.utc).isoformat(),
            encoding="utf-8",
        )
        content = marker.read_text(encoding="utf-8")
        marker.unlink()
        assert content.startswith("Google Drive write test completed")
        record("Google Drive mount and write", "PASS", str(workshop_dir.parent))
    except Exception as exc:
        record("Google Drive mount and write", "FAIL", f"{type(exc).__name__}: {exc}")
else:
    record("Google Drive mount and write", "PASS", "Not applicable to the local fallback; test Drive separately in a browser")

In [ ]:
# Network checks are warnings rather than failures because institutional networks
# may filter automated requests even when the same site opens in a browser.
URLS = {
    "Google Colab": "https://colab.research.google.com/",
    "Google Drive": "https://drive.google.com/",
    "GitHub": "https://github.com/",
    "Rhino downloads": "https://www.rhino3d.com/download/",
}

for name, url in URLS.items():
    try:
        request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request, timeout=10) as response:
            status_code = getattr(response, "status", 200)
        if status_code < 400:
            record(f"Website access: {name}", "PASS", f"HTTP {status_code}")
        else:
            record(f"Website access: {name}", "WARN", f"HTTP {status_code}; also test in your browser")
    except Exception as exc:
        record(f"Website access: {name}", "WARN", f"Automated check failed ({type(exc).__name__}); test {url} in your browser")

In [ ]:
print("\n" + "=" * 80)
print("FINAL ENVIRONMENT REPORT")
print("=" * 80)
for item in RESULTS:
    print(f"[{item['status']:<4}] {item['check']}: {item['detail']}")

failures = [item for item in RESULTS if item["status"] == "FAIL"]
warnings = [item for item in RESULTS if item["status"] == "WARN"]

print("-" * 80)
if failures:
    FINAL_STATUS = "NOT READY"
    print(f"❌ {FINAL_STATUS}: {len(failures)} required check(s) failed.")
    print("Read guides/Workshop_Setup_Guide_EN.md or the Chinese guide, fix the failures, restart the kernel, and run all cells again.")
elif warnings:
    FINAL_STATUS = "READY WITH WARNINGS"
    print(f"⚠️ {FINAL_STATUS}: no required check failed, but review {len(warnings)} warning(s).")
else:
    FINAL_STATUS = "READY"
    print("✅ READY: all checks passed.")

print("Machine-learning training performed: NO")
print("Environment check completed:", datetime.now(timezone.utc).isoformat())